Atividade — Ajuste Fino de Modelos de Geração

Preencha os campos marcados com `# 👈` e execute cada célula para ver o resultado.

In [ ]:
# ── SETUP — Execute esta célula primeiro ─────────────────────
#
# Pacotes instalados:
#   transformers  → biblioteca da Hugging Face para carregar modelos de linguagem
#   accelerate    → distribui o modelo entre CPU/GPU automaticamente
#   torch         → biblioteca de deep learning que executa os cálculos dos modelos

import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "accelerate", "torch"], check=True)

# AutoTokenizer        → divide texto em tokens (o "vocabulário" que o modelo entende)
# AutoModelForCausalLM → carrega o modelo gerador — o cérebro que prevê o próximo token
# torch                → gerencia tensores e a GPU durante a inferência
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ── Carregando os modelos para a Questão 2 ───────────────────
# Usamos o SmolLM2-360M: modelo pequeno (~360M parâmetros, ~720MB cada)
# Carregamos AGORA para não precisar baixar nada durante a atividade.

BASE    = "HuggingFaceTB/SmolLM2-360M"           # modelo base (sem fine-tuning)
INSTRUCT = "HuggingFaceTB/SmolLM2-360M-Instruct" # modelo com instrução (após SFT)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("⏳ Baixando e carregando modelo BASE...")
tok_base   = AutoTokenizer.from_pretrained(BASE)
model_base = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype, device_map="auto")
model_base.eval()

print("⏳ Baixando e carregando modelo INSTRUÇÃO (SFT)...")
tok_instruct   = AutoTokenizer.from_pretrained(INSTRUCT)
model_instruct = AutoModelForCausalLM.from_pretrained(INSTRUCT, dtype=dtype, device_map="auto")
model_instruct.eval()

print()
print("✅ Tudo pronto! Modelos carregados na memória.")
print(f"   GPU disponível: {'Sim ✅' if torch.cuda.is_available() else 'Não — CPU mode'}")


⏳ Baixando e carregando modelo BASE...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

⏳ Baixando e carregando modelo INSTRUÇÃO (SFT)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


✅ Tudo pronto! Modelos carregados na memória.
   GPU disponível: Não — CPU mode


---
## 📌 Questão 1 — LoRA: Quantos parâmetros precisamos treinar?

No **fine-tuning completo**, todos os pesos são atualizados: uma matriz $d \times d$ tem $d^2$ parâmetros.

O **LoRA** usa duas matrizes menores no lugar:
```
A: d × r   →   d * r parâmetros
B: r × d   →   r * d parâmetros
Total LoRA  =   2 * d * r
```
Os pesos originais **ficam congelados** — só A e B são treinados.

Complete a fórmula e escolha um rank para ver a comparação.

In [ ]:
# ── Questão 1 ────────────────────────────────────────────────
d = 1024  # dimensão da matriz (em modelos reais: 768, 1024, etc.)

# Escolha um rank r (valores comuns: 2, 4, 8, 16)
r = 16  # 👈 substitua None por um número inteiro (ex: 4)

# Fine-tuning completo — já calculado para você
params_full = d * d

# Calcule os parâmetros do LoRA
# Dica: A tem (d × r) parâmetros, B tem (r × d) → total = 2 * d * r
params_lora = 2 * d * r  # 👈 substitua None pela fórmula correta

# ── Resultado ────────────────────────────────────────────────
if r is None or params_lora is None:
    print("❌ Preencha os campos marcados com 👈 e execute novamente.")
else:
    print(f"Fine-tuning completo : {params_full:,} parâmetros")
    print(f"LoRA  (r={r})         : {params_lora:,} parâmetros")
    print(f"\n🚀 O LoRA é {params_full/params_lora:.0f}x mais eficiente!")
    print(f"   Treina apenas {params_lora/params_full*100:.1f}% dos parâmetros originais.")

Fine-tuning completo : 1,048,576 parâmetros
LoRA  (r=16)         : 32,768 parâmetros

🚀 O LoRA é 32x mais eficiente!
   Treina apenas 3.1% dos parâmetros originais.


---
## 📌 Questão 2 — SFT: Modelo base vs. modelo com instrução

Um modelo **base** foi treinado apenas para prever o próximo token — ele não sabe o que é uma pergunta ou uma resposta.

Com o **SFT** (Supervised Fine-Tuning), o modelo aprende a responder instruções usando dados num **chat template**. O SmolLM2 usa o formato ChatML:
```
<|im_start|>user
mensagem do usuário<|im_end|>
<|im_start|>assistant
```

Sua tarefa:
1. Escreva uma instrução em `usuario`
2. Complete a função `formatar_instrucao` com a f-string do ChatML
3. Execute e veja a diferença entre os dois modelos


In [ ]:
# ── Questão 2 ────────────────────────────────────────────────

# Escreva uma instrução simples (qualquer tema, em inglês)
# Ex: "What is photosynthesis?", "Tell me a fun fact.", "What is 2+2?"
usuario = "What is 2+2+2?" # 👈 substitua por uma string com sua instrução


def formatar_instrucao(mensagem):
    """
    Formata uma instrução no ChatML — template do SmolLM2.
    O modelo vai gerar a partir de <|im_start|>assistant.

    Formato:
        <|im_start|>user\n{mensagem}<|im_end|>\n<|im_start|>assistant\n

    Dica: f"<|im_start|>user\n{mensagem}<|im_end|>\n<|im_start|>assistant\n"
    """
    return f"<|im_start|>user\n{mensagem}<|im_end|>\n<|im_start|>assistant\n"  # 👈 substitua None pela f-string correta


# ── Resultado ────────────────────────────────────────────────
if usuario is None:
    print("❌ Preencha a variável usuario com uma string.")
elif formatar_instrucao(usuario) is None:
    print("❌ Complete a função formatar_instrucao() com a f-string.")
else:
    template  = formatar_instrucao(usuario)
    MAX_TOK   = 60

    # ── Modelo BASE: recebe a instrução crua (sem template) ──
    inp = tok_base(usuario, return_tensors="pt").to(model_base.device)
    with torch.no_grad():
        out = model_base.generate(
            **inp, max_new_tokens=MAX_TOK, do_sample=False,
            pad_token_id=tok_base.eos_token_id
        )
    saida_base = tok_base.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

    # ── Modelo INSTRUÇÃO: recebe o ChatML formatado ──────────
    inp = tok_instruct(template, return_tensors="pt").to(model_instruct.device)
    with torch.no_grad():
        out = model_instruct.generate(
            **inp, max_new_tokens=MAX_TOK, do_sample=False,
            pad_token_id=tok_instruct.eos_token_id
        )
    saida_instruct = tok_instruct.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

    # ── Comparação ───────────────────────────────────────────
    print(f'Instrução: "{usuario}"')
    print()
    print("Modelo BASE (sem fine-tuning) — só continua o texto:")
    print("-" * 45)
    print(saida_base.strip())
    print()
    print("Modelo INSTRUÇÃO (após SFT) — responde como assistente:")
    print("-" * 45)
    print(saida_instruct.strip())


Instrução: "What is 2+2+2?"

Modelo BASE (sem fine-tuning) — só continua o texto:
---------------------------------------------
The answer is 6.

## What is 2+2+2 in math?

The answer is 6.

## What is 2+2+2 in math?

The answer is 6.

## What is 2+

Modelo INSTRUÇÃO (após SFT) — responde como assistente:
---------------------------------------------
The expression 2+2+2 is a classic example of a mathematical expression that can be evaluated using basic arithmetic operations. To evaluate this expression, we can follow the order of operations, which is often remembered by the acronym PEMDAS: Parentheses, Exponents, Multiplication and Division


## 📌 Questão 3 — Preference Tuning: montando um dataset de preferência

No **Preference Tuning**, o modelo aprende a partir de pares de respostas para o mesmo prompt:

- `chosen` → resposta **preferida** (mais completa, útil, honesta)
- `rejected` → resposta **preterida** (genérica, curta, ou incorreta)

> ⚠️ Não é necessariamente **bom vs. ruim** — pode ser **bom vs. melhor**.

Cada exemplo tem o formato:
```python

In [ ]:
# ── Questão 3 ────────────────────────────────────────────────
# Use inglês para melhores resultados com o TinyLlama.
# Dica: o "chosen" deve ser mais completo/detalhado que o "rejected".

dataset_manual = [
    {
        "prompt":   None,  # 👈 substitua por uma string com uma pergunta
        "chosen":   None,  # 👈 substitua por uma resposta boa e detalhada
        "rejected": None,  # 👈 substitua por uma resposta curta ou genérica
    },
    {
        "prompt":   None,  # 👈 substitua por outra pergunta (tema diferente)
        "chosen":   None,  # 👈 resposta preferida
        "rejected": None,  # 👈 resposta rejeitada
    },
]


def formatar_par_dpo(exemplo):
    """
    Formata um exemplo de preferência no chat template do TinyLlama.

    Retorna um dicionário com as chaves "prompt", "chosen", "rejected"
    formatadas conforme o template abaixo:

        prompt_fmt   = f"<|user|>\n{exemplo['prompt']}</s>\n<|assistant|>\n"
        chosen_fmt   = f"{exemplo['chosen']}</s>\n"
        rejected_fmt = f"{exemplo['rejected']}</s>\n"
    """
    prompt_fmt   = None  # 👈 substitua pela f-string do prompt
    chosen_fmt   = None  # 👈 substitua pela f-string do chosen
    rejected_fmt = None  # 👈 substitua pela f-string do rejected

    return {"prompt": prompt_fmt, "chosen": chosen_fmt, "rejected": rejected_fmt}


# ── Validação ────────────────────────────────────────────────
erros = []
for i, ex in enumerate(dataset_manual):
    for campo in ("prompt", "chosen", "rejected"):
        if ex[campo] is None:
            erros.append(f"  Exemplo {i+1}: campo '{campo}' ainda é None.")

resultado = formatar_par_dpo({"prompt": "test", "chosen": "a", "rejected": "b"})
if resultado["prompt"] is None:
    erros.append("  formatar_par_dpo() ainda retorna None — preencha as f-strings.")

if erros:
    print("❌ Preencha os campos marcados com 👈:")
    for e in erros:
        print(e)
else:
    print("✅ Dataset de preferência criado!\n")
    for i, ex in enumerate(dataset_manual):
        fmt = formatar_par_dpo(ex)
        print(f"── Exemplo {i+1} ──────────────────────────────")
        print("PROMPT   →", repr(fmt["prompt"]))
        print("CHOSEN   →", repr(fmt["chosen"]))
        print("REJECTED →", repr(fmt["rejected"]))
        print()
    print("💡 Este é exatamente o formato esperado pelo DPOTrainer (biblioteca trl).")
    print("   O modelo aprende: P(chosen | prompt) > P(rejected | prompt)")

## 📌 Questão 4 — QLoRA: quantização e memória

O QLoRA combina duas ideias:
   1. Quantização: os pesos do modelo base são armazenados em 4 bits (INT4)
      em vez de 16 bits (FP16) — reduz a memória ~4x.
   2. LoRA: os adaptadores A e B continuam em FP16 (precisão normal).

 Fórmulas de memória (em bytes):

*   Modelo FP16  → n_params * 2          (2 bytes por parâmetro)
*   Modelo INT4  → n_params * 0.5        (0.5 bytes por parâmetro)
*   Adaptadores  → 2 * d * r * 2         (sempre FP16)

 1 GB = 1_073_741_824 bytes


In [ ]:
# ── Questão 4 ─────────────────────────────────

# Parâmetros do modelo (em quantidade de parâmetros)
n_params = 7_000_000_000  # modelo de 7 bilhões de parâmetros (ex: Llama-2-7B)

# Dimensão e rank do LoRA
d = 4096  # dimensão das camadas do modelo
r = 8     # rank escolhido

# ── Calcule a memória do modelo completo em FP16 ──────────────
mem_fp16_bytes = n_params * 2  # 👈 substitua pela fórmula do Modelo FP16

# ── Calcule a memória do modelo quantizado em INT4 ────────────
mem_int4_bytes = n_params * 0.5  # 👈 substitua pela fórmula do Modelo INT4

# ── Calcule a memória dos adaptadores LoRA em FP16 ────────────

mem_lora_bytes = 2 * d * r * 2  # 👈 substitua: pela fórmula dos Adaptadores

# ── Resultado ─────────────────────────────────────────────────
if mem_fp16_bytes is None or mem_int4_bytes is None or mem_lora_bytes is None:
    print("❌ Preencha os campos marcados com 👈 e execute novamente.")
else:
    GB = 1_073_741_824
    mem_qlora_total = mem_int4_bytes + mem_lora_bytes

    print(f"Modelo completo  FP16 : {mem_fp16_bytes  / GB:.2f} GB")
    print(f"Modelo base      INT4 : {mem_int4_bytes  / GB:.2f} GB  (quantizado)")
    print(f"Adaptadores LoRA FP16 : {mem_lora_bytes  / GB:.4f} GB")
    print(f"─────────────────────────────────────────")
    print(f"QLoRA total           : {mem_qlora_total / GB:.2f} GB")
    print(f"\n🚀 O QLoRA usa {mem_fp16_bytes / mem_qlora_total:.1f}x menos memória que o fine-tuning completo em FP16!")
    print(f"   Os adaptadores representam apenas {mem_lora_bytes / mem_qlora_total * 100:.2f}% da memória total.")
    print()
    print("💡 É por isso que o QLoRA permite fazer fine-tuning de modelos de 7B+")
    print("   em GPUs de consumo (ex: RTX 3090 com 24 GB de VRAM).")
    print("   Os pesos originais ficam congelados em INT4 — apenas A e B são treinados.")

Modelo completo  FP16 : 13.04 GB
Modelo base      INT4 : 3.26 GB  (quantizado)
Adaptadores LoRA FP16 : 0.0001 GB
─────────────────────────────────────────
QLoRA total           : 3.26 GB

🚀 O QLoRA usa 4.0x menos memória que o fine-tuning completo em FP16!
   Os adaptadores representam apenas 0.00% da memória total.

💡 É por isso que o QLoRA permite fazer fine-tuning de modelos de 7B+
   em GPUs de consumo (ex: RTX 3090 com 24 GB de VRAM).
   Os pesos originais ficam congelados em INT4 — apenas A e B são treinados.


## 📌 Questão 5 — LoRA configuration: escolhendo os hiperparâmetros

 Na prática, o LoRA é configurado com um objeto LoraConfig (biblioteca peft).
 Os principais hiperparâmetros são:


*   r → rank das matrizes A e B (controla capacidade vs. eficiência)

    valores comuns: **4**, 8, 16, 32
*   lora_alpha → escala aplicada aos adaptadores na soma com os pesos originais
    
    a atualização efetiva é: W + (alpha/r) * B*A
    
    convenção comum: lora_alpha = **2 * r**
*   target_modules → quais camadas recebem adaptadores LoRA
                    
    opções típicas:
    
    ["q_proj", "v_proj"]          (mínimo)

    ["q_proj", "k_proj", "v_proj", "o_proj"]  (completo)

*   lora_dropout → dropout aplicado nas matrizes A e B (regularização)
    
    valor comum: **0.05**


 Nesta questão você vai preencher uma configuração LoRA e calcular
 o scaling factor (**lora_alpha / r**), que determina a intensidade da adaptação.

In [ ]:
# ── Questão 5 ──────────────────────────────────────────────────

# ── Escolha seus hiperparâmetros ─────────────────────────────
r           = 4   # 👈 substitua por um valor comum de r
lora_alpha  = 2 * r   # 👈 substitua pela convenção comum de lora_alpha
lora_dropout = 0.05  # 👈 substitua por um valor comum de lora_dropout

# ── Escolha os módulos-alvo ───────────────────────────────────
# Opção A — mínimo  : ["q_proj", "v_proj"]
# Opção B — completo: ["q_proj", "k_proj", "v_proj", "o_proj"]
target_modules = ["q_proj", "v_proj"]  # 👈 substitua por uma das listas acima

# ── Calcule o scaling factor ──────────────────────────────────
scaling = lora_alpha / r  # 👈 substitua pela fórmula correta

# ── Resultado ─────────────────────────────────────────────────
if any(x is None for x in [r, lora_alpha, lora_dropout, target_modules, scaling]):
    print("❌ Preencha os campos marcados com 👈 e execute novamente.")
else:
    print("✅ Configuração LoRA definida!\n")
    print(f"  r              = {r}")
    print(f"  lora_alpha     = {lora_alpha}")
    print(f"  lora_dropout   = {lora_dropout}")
    print(f"  target_modules = {target_modules}")
    print(f"  scaling factor = alpha/r = {lora_alpha}/{r} = {scaling:.2f}")
    print()

    if scaling == 1.0:
        interpretacao = "neutro — adaptação na mesma escala dos pesos originais."
    elif scaling > 1.0:
        interpretacao = "amplificado — adaptadores têm mais influência que o padrão."
    else:
        interpretacao = "atenuado — adaptadores têm menos influência que o padrão."

    print(f"  📐 Scaling {interpretacao}")
    print()
    print("  Equivalente em código (biblioteca peft):")
    print()
    print("  from peft import LoraConfig")
    print("  config = LoraConfig(")
    print(f"      r               = {r},")
    print(f"      lora_alpha      = {lora_alpha},")
    print(f"      lora_dropout    = {lora_dropout},")
    print(f"      target_modules  = {target_modules},")
    print( "      bias            = 'none',")
    print( "      task_type       = 'CAUSAL_LM',")
    print( "  )")
    print()
    print("💡 r alto → mais capacidade, mais parâmetros treináveis.")
    print("   r baixo → mais eficiente, pode ser insuficiente para tarefas complexas.")
    print("   O scaling (alpha/r) evita reescalar manualmente o lr ao mudar r.")

✅ Configuração LoRA definida!

  r              = 4
  lora_alpha     = 8
  lora_dropout   = 0.05
  target_modules = ['q_proj', 'v_proj']
  scaling factor = alpha/r = 8/4 = 2.00

  📐 Scaling amplificado — adaptadores têm mais influência que o padrão.

  Equivalente em código (biblioteca peft):

  from peft import LoraConfig
  config = LoraConfig(
      r               = 4,
      lora_alpha      = 8,
      lora_dropout    = 0.05,
      target_modules  = ['q_proj', 'v_proj'],
      bias            = 'none',
      task_type       = 'CAUSAL_LM',
  )

💡 r alto → mais capacidade, mais parâmetros treináveis.
   r baixo → mais eficiente, pode ser insuficiente para tarefas complexas.
   O scaling (alpha/r) evita reescalar manualmente o lr ao mudar r.


## 📌 Questão 5 — Reward Model: simulando a pontuação de respostas

O **Reward Model** é o coração do RLHF clássico com PPO.

Ele recebe um par `(prompt, resposta)` e devolve **um único número** — quanto maior o score, melhor a resposta. É treinado com o objetivo simples:

$$\text{score(chosen)} > \text{score(rejected)}$$

Na prática, o Reward Model é uma cópia do LLM instruction-tuned com a **cabeça de geração de texto substituída por uma cabeça de classificação** que emite esse score.

Nesta questão você vai **simular** esse objetivo implementando um reward heurístico — uma versão simplificada com regras manuais. Isso te ajuda a entender o que o Reward Model real aprende automaticamente.

Implemente **pelo menos 3** das heurísticas sugeridas no código.


In [ ]:
# ── Questão 5 ────────────────────────────────────────────────

def reward_heuristico(resposta: str) -> float:
    """
    Atribui um score de qualidade a uma resposta de texto.
    Retorne um float — quanto maior, melhor a resposta.

    Implemente pelo menos 3 das heurísticas abaixo:

      1. Comprimento: respostas mais longas tendem a ser mais completas.
         score += len(resposta) * 0.01

      2. Pontuação: respostas bem escritas têm vírgulas e pontos.
         score += resposta.count('.') * 0.5

      3. Início com maiúscula (sinal de boa escrita).
         if len(resposta) > 0 and resposta[0].isupper(): score += 1.0

      4. Penalizar respostas muito curtas (menos de 20 caracteres).
         if len(resposta) < 20: score -= 2.0

      5. Presença de conectivos explicativos.
         conectivos = ["because", "therefore", "however", "which means"]
         score += sum(1 for w in conectivos if w in resposta.lower())
    """
    score = 0.0

    # 👈 Implemente suas heurísticas aqui (mínimo 3)

    return score


# ── Pares de teste ────────────────────────────────────────────
pares_teste = [
    {
        "prompt":   "What is a neural network?",
        "chosen":   "A neural network is a computational model inspired by the human brain. It consists of layers of interconnected nodes that process and transform data. Neural networks learn by adjusting their weights during training to minimize prediction errors.",
        "rejected": "It's a network.",
    },
    {
        "prompt":   "Explain gradient descent.",
        "chosen":   "Gradient descent is an optimization algorithm used to minimize a loss function. It works by computing the gradient of the loss with respect to model parameters and updating them in the opposite direction, therefore reducing the error iteratively.",
        "rejected": "gradient descent makes model learn",
    },
]

# ── Validação ────────────────────────────────────────────────
if reward_heuristico("This is a test sentence.") == 0.0 and reward_heuristico("x") == 0.0:
    print("❌ Implemente pelo menos 3 heurísticas em reward_heuristico() (campo com 👈).")
else:
    print("✅ Reward heurístico implementado!\n")
    acertos = 0
    for i, par in enumerate(pares_teste):
        sc = reward_heuristico(par["chosen"])
        sr = reward_heuristico(par["rejected"])
        ok = sc > sr
        if ok:
            acertos += 1
        status = "✅" if ok else "❌"
        print(f"Par {i+1}: {status}  score(chosen)={sc:.2f}  |  score(rejected)={sr:.2f}")
        if not ok:
            print("        ⚠ Objetivo score(chosen) > score(rejected) NÃO satisfeito.")
            print("          Revise suas heurísticas para penalizar respostas curtas/incompletas.")

    print(f"\nResultado: {acertos}/{len(pares_teste)} pares corretos")
    print()
    print("💡 Um Reward Model REAL aprende essas preferências automaticamente,")
    print("   treinado em milhares de pares anotados por humanos (como no Llama 2).")
    print("   O DPO elimina essa etapa usando o próprio LLM como juiz.")

## 📌 Questão 6 — DPO: o parâmetro beta e a função de perda

O **DPO (Direct Preference Optimization)** não treina um Reward Model separado. Em vez disso, compara as probabilidades token a token entre duas versões do modelo:

| Modelo | Papel |
|--------|-------|
| `modelo_ref` | Cópia **congelada** — ponto de partida, não muda |
| `modelo_treino` | LLM sendo atualizado durante o treino |

O **log_ratio** mede o quanto o modelo treinável se afastou da referência para uma dada resposta:

$$\text{log\_ratio} = \log P(\text{resposta} \mid \text{treino}) - \log P(\text{resposta} \mid \text{ref})$$

A função de perda do DPO é:

$$\mathcal{L} = -\log\left(\sigma\left(\beta \cdot (\text{log\_ratio\_chosen} - \text{log\_ratio\_rejected})\right)\right)$$

O parâmetro **beta** controla o desvio permitido da referência:
- `beta` alto (ex: `1.0`) → modelo fica próximo da referência *(conservador)*
- `beta` baixo (ex: `0.1`) → modelo pode mudar mais livremente *(agressivo)*

No paper original do DPO, `beta=0.1` produziu os melhores resultados.

Nesta questão você vai implementar essa função e observar como o `beta` influencia a perda em três cenários de treinamento.


In [ ]:
# ── Questão 6 ────────────────────────────────────────────────
import math

def dpo_loss(log_ratio_chosen: float, log_ratio_rejected: float, beta: float) -> float:
    """
    Calcula a perda DPO para um único par (chosen, rejected).

    Parâmetros:
      log_ratio_chosen   → log P(chosen|treino) - log P(chosen|ref)
                           positivo = modelo treino mais confiante que ref no chosen
      log_ratio_rejected → log P(rejected|treino) - log P(rejected|ref)
                           negativo = modelo treino menos confiante que ref no rejected
      beta               → controla o desvio permitido da referência

    Fórmula (implemente em 2 linhas):
      margin = beta * (log_ratio_chosen - log_ratio_rejected)
      loss   = -math.log(1 / (1 + math.exp(-margin)))
    """
    margin = None  # 👈 calcule: beta * (log_ratio_chosen - log_ratio_rejected)
    loss   = None  # 👈 calcule: -math.log(1 / (1 + math.exp(-margin)))

    return loss


# Escolha um valor de beta para experimentar.
# Valores comuns no paper: 0.1, 0.5, 1.0
beta = None  # 👈 substitua por um número float (ex: 0.1)


# ── Validação ────────────────────────────────────────────────
if beta is None:
    print("❌ Defina o valor de beta (campo com 👈).")
elif dpo_loss(1.0, -1.0, beta) is None:
    print("❌ Complete a função dpo_loss() (campos com 👈).")
else:
    print(f"✅ DPO loss implementada! beta = {beta}\n")

    cenarios = [
        {"nome": "Ideal   — modelo melhora bastante", "chosen": +1.5, "rejected": -1.5},
        {"nome": "Neutro  — modelo não mudou",        "chosen":  0.0, "rejected":  0.0},
        {"nome": "Invertido — modelo piorou",         "chosen": -1.0, "rejected": +1.0},
    ]

    print(f"{'Cenário':<42} {'log_ratio_chosen':>17} {'log_ratio_rejected':>19} {'Loss':>8}")
    print("─" * 90)
    for c in cenarios:
        loss = dpo_loss(c["chosen"], c["rejected"], beta)
        print(f"{c['nome']:<42} {c['chosen']:>17.2f} {c['rejected']:>19.2f} {loss:>8.4f}")

    print()
    print("💡 Quanto menor a loss, melhor o modelo está aprendendo.")
    comportamento = "resiste mais a mudanças (conservador)" if beta >= 0.5 else "pode se afastar mais da referência (agressivo)"
    print(f"   Com beta={beta}, o modelo {comportamento}.")
    print()
    print("   O beta é o 'freio' do DPO — evita que o modelo esqueça")
    print("   o que aprendeu no SFT ao ser alinhado com preferências.")